### Random vs Random
- First goal of this analytics is to decide number of maximum steps before truncation for training for 4x5 board.
 - Second goal is to show if there is any bias towards specific color from domain and application.


In [5]:
import time
from collections import Counter
from environment.grenight_environment import GrenightEnvironment
from domain.configs import MAX_ALLOWED_STEPS_WITHOUT_PAWN_MOVE_OR_CAPTURING

In [6]:
MAX_MOVES_PER_EPISODES = [25, 40, 50, 60, 70, 80, 90, 100, 200, 500]

In [7]:
def play_random_game(env_arg: GrenightEnvironment,
                     current_max_moves_per_ep: int) -> tuple[str, int, dict]:

    env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < current_max_moves_per_ep:
        acting_player_is_white = env_arg.is_white_on_turn
        action = env_arg.sample()
        _, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", move_count, info
    if reward == 0.0:
        return "draw", move_count, info

    winner_is_white = acting_player_is_white if reward > 0 else not acting_player_is_white

    return ("white_win", move_count, info) if winner_is_white else ("black_win", move_count, info)

In [8]:
env = GrenightEnvironment()

for current_max_steps_per_ep in MAX_MOVES_PER_EPISODES:

    outcomes_counter = Counter()
    total_step_counts_per_outcome = Counter()
    draw_reasons = Counter()

    start_time = time.perf_counter()
    for _ in range(100):
        outcome, steps, game_info = play_random_game(env, current_max_steps_per_ep)
        outcomes_counter[outcome] += 1
        if outcome != "truncated":
            total_step_counts_per_outcome[outcome] += steps
        if game_info["draw_reason"] is not None:
            draw_reasons[game_info["draw_reason"]] += 1
    end_time = time.perf_counter()

    avg_step_counts_per_outcome = dict()
    for outcome, total_steps in total_step_counts_per_outcome.items():
        avg_step_counts_per_outcome[outcome] = total_steps / outcomes_counter[outcome]

    print(f"Outcomes in 100 games where max steps without pawn movement or capturing: {MAX_ALLOWED_STEPS_WITHOUT_PAWN_MOVE_OR_CAPTURING}\n"
          f"And total max steps per episode was {current_max_steps_per_ep}:\n"
          f"{outcomes_counter}\n"
          f"Average moves per outcome: \n{avg_step_counts_per_outcome}\n"
          f"Draw reasons: \n{draw_reasons}\n"
          f"Execution time: {end_time - start_time:.2f} seconds\n")

Outcomes in 100 games where max steps without pawn movement or capturing: 30
And total max steps per episode were 25:
Counter({'truncated': 81, 'white_win': 8, 'draw': 6, 'black_win': 5})
Average moves per outcome: 
{'white_win': 17.5, 'draw': 19.833333333333332, 'black_win': 17.2}
Draw reasons: 
Counter({'stalemate': 6})
Execution time: 12.76 seconds

Outcomes in 100 games where max steps without pawn movement or capturing: 30
And total max steps per episode were 40:
Counter({'truncated': 71, 'black_win': 15, 'white_win': 8, 'draw': 6})
Average moves per outcome: 
{'draw': 27.166666666666668, 'black_win': 17.2, 'white_win': 23.25}
Draw reasons: 
Counter({'stalemate': 6})
Execution time: 16.77 seconds

Outcomes in 100 games where max steps without pawn movement or capturing: 30
And total max steps per episode were 50:
Counter({'truncated': 54, 'draw': 23, 'black_win': 12, 'white_win': 11})
Average moves per outcome: 
{'draw': 38.08695652173913, 'white_win': 17.545454545454547, 'black_w

#### Conclusion
- From this analytics for 4x5 board I am deciding that 200 is correct number for maximum steps before truncation in training.
- Out of this few hundred games total difference of winner color is not drastic. There is no correlation between maximum steps before truncation and which color had more wins. There is no correlation between average steps to win and which color had more wins over the number of maximum steps before truncation.